# 02 - Exploratory Data Analysis (EDA)
## Global Superstore Dataset (2011-2014)

**الهدف:** فهم توزيع المتغيرات والعلاقات بينها قبل بناء الـ KPIs، عبر ثلاث مستويات:
Univariate -> Bivariate -> Multivariate.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '../src')

sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/cleaned_superstore.csv', parse_dates=['Order Date', 'Ship Date'])
df['Year'] = df['Order Date'].dt.year
df.head()

## 1. Univariate Analysis
### 1.1 Numerical Variables

**السؤال:** إزاي القيم موزعة؟ فيه انحراف (Skewness)؟

In [ ]:
num_cols = ['Sales', 'Profit', 'Discount', 'Quantity', 'Shipping Cost', 'Processing Time']
summary = df[num_cols].describe().T
summary['skew'] = df[num_cols].skew()
print(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(df['Sales'], bins=60, ax=ax)
ax.set_title('Sales Distribution (Right-Skewed)')
plt.show()

### 1.2 Categorical Variables

**السؤال:** إيه أكتر الفئات تكرارًا، وهل التوزيع متوازن؟

In [ ]:
categorical_cols = ['Category','Sub-Category','Segment','Market','Region','Ship Mode','Order Priority']
for col in categorical_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts(normalize=True).mul(100).round(1))
    print()

### 1.3 Temporal Distribution

**السؤال:** هل عدد الطلبات بيزيد مع الوقت؟

In [ ]:
print(df['Year'].value_counts().sort_index())

## 2. Bivariate Analysis
### 2.1 Category vs Sales & Profit

In [ ]:
cat = df.groupby('Category').agg(
    Total_Sales=('Sales','sum'), Total_Profit=('Profit','sum'), Orders=('Sales','count')
)
cat['Profit_Margin_%'] = (cat['Total_Profit']/cat['Total_Sales']*100).round(2)
print(cat)

### 2.2 Sub-Category Deep Dive

مين بالظبط جوه Furniture بيسحب الهامش لتحت؟

In [ ]:
sub = df.groupby('Sub-Category').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
sub['Margin_%'] = (sub['Profit']/sub['Sales']*100).round(1)
print(sub.sort_values('Sales', ascending=False))

### 2.3 Discount vs Profit — أهم علاقة في المشروع

In [ ]:
correlation = df[['Discount','Profit']].corr().iloc[0,1]
print('Correlation:', round(correlation,3))

bins = [-0.01,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.85]
labels = ['0%','0-10%','10-20%','20-30%','30-40%','40-50%','50-60%','60-70%','70-85%']
df['Discount_Bucket'] = pd.cut(df['Discount'], bins=bins, labels=labels)
avg_profit = df.groupby('Discount_Bucket', observed=True)['Profit'].mean().round(1)
print(avg_profit)

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
colors = ['#d62728' if v < 0 else '#2ca02c' for v in avg_profit.values]
avg_profit.plot(kind='bar', ax=ax, color=colors)
ax.set_title('Average Profit per Order by Discount Range')
ax.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45)
plt.show()

### 2.4 Market vs Profit Margin

In [ ]:
mk = df.groupby('Market').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
mk['Margin_%'] = (mk['Profit']/mk['Sales']*100).round(1)
print(mk.sort_values('Margin_%'))

### 2.5 Segment vs Performance

In [ ]:
seg = df.groupby('Segment').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'),
    AOV=('Sales','mean'), Orders=('Sales','count')
)
seg['Margin_%'] = (seg['Profit']/seg['Sales']*100).round(1)
print(seg)

### 2.6 Yearly Trend

In [ ]:
yr = df.groupby('Year').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
yr['Margin_%'] = (yr['Profit']/yr['Sales']*100).round(1)
print(yr)

### 2.7 Validation: Order Priority vs Processing Time

In [ ]:
print(df.groupby('Order Priority')['Processing Time'].mean().round(2))

## 3. Multivariate Analysis
### 3.1 Tables x Market x Discount

هل مشكلة Tables موجودة في كل الأسواق بنفس الشكل؟

In [ ]:
tables = df[df['Sub-Category']=='Tables']
print('Tables avg discount:', round(tables['Discount'].mean(),3), 'vs company:', round(df['Discount'].mean(),3))

tm = tables.groupby('Market').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'),
    Avg_Discount=('Discount','mean'), Orders=('Sales','count')
)
tm['Margin_%'] = (tm['Profit']/tm['Sales']*100).round(1)
print(tm.sort_values('Profit'))

### 3.2 EMEA x Category x Discount

هل ضعف EMEA بسبب منتج معين، ولا مشكلة نظامية؟

In [ ]:
emea = df[df['Market']=='EMEA']
print('EMEA avg discount:', round(emea['Discount'].mean(),3), 'vs company:', round(df['Discount'].mean(),3))

ecat = emea.groupby('Category').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
ecat['Margin_%'] = (ecat['Profit']/ecat['Sales']*100).round(1)
print(ecat)

print()
print(df.groupby('Market')['Discount'].mean().round(3).sort_values(ascending=False))

### 3.3 Ship Mode / Processing Time x Profit

فرضية: هل الشحن السريع بياكل من الهامش؟

In [ ]:
ship_analysis = df.groupby('Ship Mode').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'),
    Avg_Ship_Cost=('Shipping Cost','mean'), Orders=('Sales','count')
)
ship_analysis['Margin_%'] = (ship_analysis['Profit']/ship_analysis['Sales']*100).round(1)
print(ship_analysis)

print()
print('Corr Processing Time - Profit:', round(df['Processing Time'].corr(df['Profit']),3))
print('Corr Shipping Cost - Profit:', round(df['Shipping Cost'].corr(df['Profit']),3))

## EDA Summary

1. Sales/Profit/Shipping Cost منحرفة بشدة (Right-Skewed)
2. Furniture أضعف Category ربحيًا (6.94%) بسبب Tables تحديدًا
3. نقطة انهيار الخصم عند 20%
4. EMEA أضعف Market (5.4%) بسبب خصم أعلى نظاميًا في كل الفئات
5. Segment / Ship Mode مش عوامل مؤثرة في الربحية
6. النمو الزمني صحي والهامش العام مستقر